# Step 4: The comparison models (Claude Haiku, Claude Sonnet, Jev)
### Neeraj, MMAI 894 (router fine-tuning)

**Goal:** score the models the tuned router has to beat, on exactly the same chats, the same input text and the same three answers as the tuned model, so the comparison is like for like.

| Model | How it is asked | Why it is here |
|---|---|---|
| Claude Haiku 5.5 | the team's router prompt (v3), unchanged, the team's settings | the router the team's system uses today |
| Claude Sonnet 5.5 | the same prompt and settings | the large model |
| Jev 1.14 (TypeSafe) | typed questions (pick one of 10 types, yes/no escalation, en/fr) built from the same prompt wording; it returns a probability for every choice | a small, cheap decision model |

- **Chats:** the 160 pool chats (the 40 test chats are not opened here; they are scored once, in the final run).
- **Inputs:** `first_message` (what a router sees in production) and `full_conversation` (what the team's router reads). The tuned model was scored on both in notebook 02.
- **Scored:** request type, escalation, language. The prompt also asks for a reason and other types; they are saved but not scored (the tuned model does not have them).
- **Repeats:** Haiku and Sonnet 3 times each (their answers vary from run to run), Jev twice.

**What you need to do:**
1. Kaggle: **Add-ons → Secrets → Add secret**, name `OPENROUTER_API_KEY`, value your OpenRouter key, and tick it for this notebook.
2. Accelerator: **None** (no GPU needed). Internet: **On**. This can run while notebook 03 trains.
3. Run all cells. About 20 to 30 minutes. **Expected cost about $6** (almost all of it Sonnet); the run stops itself at **$10**. Every answer is saved, so if the session stops, run again and it carries on without paying twice.
4. At the end, download `results_04.zip` (listed in the last cell).

Everything shown is computed from saved answers; this text holds no results.

## Step 0: Get the latest code
Fresh clone of the repo, so we run exactly what is on GitHub. No extra libraries are needed.

In [ ]:
import importlib
import os
import shutil
import sys

REPO_URL = "https://github.com/neerajmehla28/northstar-finetuning"
REPO_DIR = "/kaggle/working/northstar-finetuning"

%cd /kaggle/working
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone -q {REPO_URL} {REPO_DIR}          # download the repo (public, so no login is needed)

%cd {REPO_DIR}
sys.path.insert(0, "src")
importlib.invalidate_caches()
for name in [m for m in sys.modules if m.startswith("northstar")]:
    del sys.modules[name]

assert os.path.exists("src/northstar/finetune/baselines.py"), "code missing: has the latest code been pushed to GitHub?"
assert os.path.exists("prompts/router_v3.md"), "the team router prompt is missing from the repo"
assert os.path.exists("results/cv_old_recipe/full_conversation"), "notebook 02's results are missing from the repo"
!git log --oneline -3

## Step 1: The key, the data and the settings
The key is read from Kaggle's secret store and never printed or saved. The prompt file is checked against the copy recorded in `prompts/README.md`.

In [ ]:
import json
import time

import numpy as np
import pandas as pd
try:
    from kaggle_secrets import UserSecretsClient
except ImportError:          # not on Kaggle
    UserSecretsClient = None

from northstar.finetune import baselines as B
from northstar.finetune import protocol as P

API_KEY = None
try:                                    # Kaggle secret; without one, the saved answers are replayed for free
    API_KEY = UserSecretsClient().get_secret("OPENROUTER_API_KEY")
except Exception:
    pass

df = P.load_all()                 # refuses to run if the dataset file is not the exact team file
pool = P.pool(df)                 # the 160 chats; the 40 test chats are not in here

PROMPT_SHA = "3d587ac96476192a40cd9335d89fab19d583e000ba355169f258c199eb312468"     # prompts/README.md
assert P.sha256_of(B.PROMPT_PATH) == PROMPT_SHA, "prompts/router_v3.md is not the recorded copy of the team prompt"

RESULTS = "/kaggle/working/results_04"
PREDS, CACHE = f"{RESULTS}/preds", f"{RESULTS}/cache"     # one CSV per model x input x repeat; one file per answer
VARIANTS = ("first_message", "full_conversation")
REPEATS = {"haiku": 3, "sonnet": 3, "jev": 2}
BUDGET_USD = 10.0                                            # the run stops itself here
client = B.Client(API_KEY, CACHE, budget_usd=BUDGET_USD)
if os.path.exists("results/step04/cache"):            # the answers saved in the repo: reused, never paid for twice
    shutil.copytree("results/step04/cache", CACHE, dirs_exist_ok=True)
print("API key:", "loaded" if API_KEY else "none, so the saved answers are replayed (free; stops at any call that was not saved)")

run_config = dict(models=B.MODELS, reasoning_effort=B.EFFORT, max_tokens=B.MAX_TOKENS, temperature="not set (provider default)",
                  prompt="prompts/router_v3.md", prompt_sha256=PROMPT_SHA, variants=VARIANTS, repeats=REPEATS,
                  budget_usd=BUDGET_USD, date=time.strftime("%Y-%m-%d"),
                  git_commit=os.popen("git rev-parse --short HEAD").read().strip(), chats=len(pool))
os.makedirs(RESULTS, exist_ok=True)
json.dump(run_config, open(f"{RESULTS}/run_config.json", "w"), indent=1)
print(json.dumps(run_config, indent=1))

## Step 2: Probe (3 chats per model) and cost estimate
Three chats per model, to prove every call works and to measure the cost per chat before the full run. These answers are kept and reused by the full run. Jev's raw answer for one chat is printed: if Jev's reply is not in the documented shape, this cell stops and the printout shows why (paste it to Claude).

In [ ]:
probe = pd.concat([B.run_variant(client, m, pool, "first_message", 0, PREDS, limit=3) for m in REPEATS], ignore_index=True)
display(probe[["id", "model", "type", "esc", "language", "reason", "confidence", "latency_ms", "cost_usd", "fallback", "error"]])

raw = client.decide(B.MODELS["jev"], P.input_text(pool, "first_message").iloc[0], B.jev_questions(), "rep0")
print("Jev reply fields:", raw["raw_keys"])
print(json.dumps(raw["answers"], indent=1)[:2500])
assert not probe["fallback"].any(), "a probe call gave no usable answer: see the error column and the printout above"

per_chat = probe.groupby("model")["cost_usd"].mean()
calls = {m: len(pool) * len(VARIANTS) * n for m, n in REPEATS.items()}
estimate = sum(per_chat[m] * calls[m] for m in REPEATS) * 1.15          # +15%: the full conversation is longer
assert estimate < BUDGET_USD, f"estimate ${estimate:.2f} is above the limit: lower REPEATS or raise BUDGET_USD in Step 1 on purpose"
print("cost per chat ($):", per_chat.round(6).to_dict())
print(f"Estimated cost of the full run: ${estimate:.2f} (limit ${BUDGET_USD:.0f})")

## Step 3: The full run
Every model, both inputs, every repeat, every pool chat. Six calls run at once to save time (speed is measured separately in Step 4). A file that is already finished is not run again, and every single answer is cached, so this cell can simply be rerun after an interruption.

In [ ]:
for m, n_rep in REPEATS.items():
    for variant in VARIANTS:
        for rep in range(n_rep):
            t0 = time.time()
            d = B.run_variant(client, m, pool, variant, rep, PREDS, workers=6)
            print(f"{m:6s} {variant:17s} repeat {rep}: {len(d)} chats | fallbacks {int(d['fallback'].sum())} | "
                  f"corrections {int((d['error'] == 'needed one correction').sum())} | ${d['cost_usd'].sum():.3f} | {time.time() - t0:.0f}s", flush=True)
print(f"New money spent in this session: ${client.spent:.2f}")

## Step 4: Speed, one call at a time
The first 40 pool chats (first message), sent one after another with nothing else running, the way one customer's chat would be routed. The first call is a warm-up and is left out. This is the network round trip from Kaggle to OpenRouter and back, so it includes internet time. The tuned model's speed is measured on a CPU in the final run.

In [ ]:
lat_rows = []
for m in REPEATS:
    lp = B.latency_probe(client, m, pool, "first_message", n=40).iloc[1:]            # drop the warm-up call
    lat_rows.append(dict(model=m, chats=len(lp), median_ms=lp["latency_ms"].median(), p90_ms=lp["latency_ms"].quantile(0.9)))
latency = pd.DataFrame(lat_rows).set_index("model")
latency.round(0).to_csv(f"{RESULTS}/latency.csv")
latency.round(0)

## Step 5: Scores, next to the tuned model
The tuned model's numbers are its out-of-fold scores from notebook 02 (original recipe, epoch 30, the same input), read from the repo. For each method: the mean ± spread of single runs (seeds for the tuned model, repeats for the others), and a **combined** answer (the tuned model averages its 3 seeds; the others average their repeats). The 95% interval comes from resampling customers. Cost is per chat as billed by OpenRouter.

In [ ]:
import matplotlib.pyplot as plt
from northstar.finetune import cv_analysis as A
from northstar.finetune import metrics as M

TUNED_EPOCH = 30                       # notebook 02's choice for the original recipe
TUNED_DIR = "results/cv_old_recipe"
y_type, y_esc, y_lang = pool["type_label"].values, pool["escalation_label"].values, pool["language_label"].values
customers = pool["customer_id"].values


def score(type_probs, esc_score, language=None):
    t, e = M.type_scores(y_type, type_probs), M.escalation_scores(y_esc, esc_score)
    return dict(type_accuracy=t["accuracy"], type_macro_f1=t["macro_f1"], esc_recall=e["recall"], esc_precision=e["precision"],
                esc_f2=e["f2"], esc_missed=e["missed"], esc_false_alarms=e["false_alarms"],
                language_accuracy=float((np.asarray(language) == y_lang).mean()) if language is not None else np.nan)


methods = {}                           # (method, input) -> single-run scores, combined answer, cost, fallbacks
for variant in VARIANTS:
    oof = A.assemble(A.load_variant(TUNED_DIR, variant), pool)
    s = A.summary_at_epoch(oof, pool, TUNED_EPOCH)
    methods[("tuned XLM-R", variant)] = dict(
        per=[score(a["type"][TUNED_EPOCH - 1], a["escalation"][TUNED_EPOCH - 1]) for a in oof.values()],
        combined=dict(type_probs=s["ensemble_type_logits"], esc_score=s["ensemble_escalation_logits"], language=None),
        cost=0.0, fallbacks=0)
    for m in REPEATS:
        reps = B.load_preds(PREDS, m, variant, pool)
        methods[(m, variant)] = dict(per=[score(**B.arrays(r)) for r in reps], combined=B.vote(reps),
                                     cost=float(np.mean([r["cost_usd"].mean() for r in reps])),
                                     fallbacks=int(sum(r["fallback"].sum() for r in reps)))

rows, numeric = [], []
for (name, variant), d in methods.items():
    per = pd.DataFrame(d["per"])
    comb = score(**d["combined"])
    correct = d["combined"]["type_probs"].argmax(-1) == y_type
    lo, hi = M.cluster_bootstrap(lambda idx, c=correct: c[idx].mean(), customers, n_boot=2000)
    ms = lambda c: f"{per[c].mean():.3f} ± {per[c].std(ddof=1) if len(per) > 1 else 0:.3f}"
    rows.append({"method": name, "input": variant, "runs": len(per), "type accuracy": ms("type_accuracy"),
                 "type macro-F1": ms("type_macro_f1"), "escalation recall": ms("esc_recall"),
                 "escalation precision": ms("esc_precision"),
                 "language accuracy": ms("language_accuracy") if per["language_accuracy"].notna().all() else "n/a (no head)",
                 "combined type acc [95% CI]": f"{comb['type_accuracy']:.3f} [{lo:.2f}, {hi:.2f}]",
                 "$ per chat": f"{d['cost']:.6f}", "fallbacks": d["fallbacks"]})
    numeric.append(dict(method=name, input=variant, runs=len(per), cost_per_chat=d["cost"], fallbacks=d["fallbacks"],
                        **{f"{c}_mean": per[c].mean() for c in per}, **{f"{c}_sd": per[c].std(ddof=1) for c in per},
                        **{f"combined_{c}": v for c, v in comb.items()}, combined_type_ci_low=lo, combined_type_ci_high=hi))
table = pd.DataFrame(rows).set_index(["method", "input"])
pd.DataFrame(numeric).to_csv(f"{RESULTS}/comparison_pool.csv", index=False)
table

## Step 6: Is any difference real?
Each comparison model against the tuned model, on the same chats and the same input (exact McNemar on the combined answers: only the chats where exactly one of the two is right count; a small p-value means the difference is unlikely to be luck). All pairs are listed.

In [ ]:
pair_rows = []
for variant in VARIANTS:
    tuned_ok = methods[("tuned XLM-R", variant)]["combined"]["type_probs"].argmax(-1) == y_type
    for m in REPEATS:
        other_ok = methods[(m, variant)]["combined"]["type_probs"].argmax(-1) == y_type
        r = M.mcnemar_exact(other_ok, tuned_ok)
        pair_rows.append(dict(input=variant, method=m, versus="tuned XLM-R", only_method_right=r["only_a_right"],
                              only_tuned_right=r["only_b_right"], p_value=r["p_value"]))
pairs = pd.DataFrame(pair_rows)
pairs.round(4).to_csv(f"{RESULTS}/paired_tests_pool.csv", index=False)
pairs.round(4)

## Step 7: Where each method wins and loses
Share of chats right for each request type and for each language (combined answers). This is where the report's "where fine-tuning wins or loses" comes from.

In [ ]:
by_type, by_lang = {}, {}
for (name, variant), d in methods.items():
    pred = d["combined"]["type_probs"].argmax(-1)
    by_type[(name, variant)] = M.per_class_recall(y_type, pred, len(P.REQUEST_TYPES))
    by_lang[(name, variant)] = [float((pred[y_lang == i] == y_type[y_lang == i]).mean()) for i in range(len(P.LANGUAGES))]
counts = pd.Series(np.bincount(y_type, minlength=len(P.REQUEST_TYPES)), index=P.REQUEST_TYPES, name="chats per type")
type_table = pd.DataFrame(by_type, index=P.REQUEST_TYPES).T
lang_table = pd.DataFrame(by_lang, index=[f"type accuracy, {l} ({(y_lang == i).sum()} chats)" for i, l in enumerate(P.LANGUAGES)]).T
type_table.round(2).to_csv(f"{RESULTS}/type_recall_by_method.csv"); lang_table.round(3).to_csv(f"{RESULTS}/type_accuracy_by_language.csv")
display(counts.to_frame().T); display(type_table.round(2)); display(lang_table.round(3))

fig, ax = plt.subplots(figsize=(9, 4.5))
labels, accs, los, his = [], [], [], []
for _, r in pd.DataFrame(numeric).iterrows():
    labels.append(f"{r['method']}\n{r['input'].replace('_', ' ')}"); accs.append(r["combined_type_accuracy"])
    los.append(r["combined_type_accuracy"] - r["combined_type_ci_low"]); his.append(r["combined_type_ci_high"] - r["combined_type_accuracy"])
ax.errorbar(range(len(labels)), accs, yerr=[los, his], fmt="o", capsize=4)
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, fontsize=7, rotation=45, ha="right")
ax.set_ylabel("request-type accuracy (combined, 95% CI)"); ax.set_title("160 pool chats: tuned model vs comparison models")
plt.tight_layout(); plt.savefig(f"{RESULTS}/comparison_pool.png", dpi=150); plt.show()

## Step 8: Save the results
The zip holds every answer (one CSV per model, input and repeat), the cache of raw replies (so anyone can replay this notebook without a key or any cost), and the tables and figure. Download `results_04.zip` and tell me.

In [ ]:
shutil.make_archive("/kaggle/working/results_04", "zip", RESULTS)
print(f"results_04.zip  {os.path.getsize('/kaggle/working/results_04.zip') / 1024:.0f} KB")
for name in sorted(os.listdir(RESULTS)):
    p = os.path.join(RESULTS, name)
    print(f"  {name:36s} {'folder, ' + str(len(os.listdir(p))) + ' files' if os.path.isdir(p) else f'{os.path.getsize(p) / 1024:.0f} KB'}")

## What this notebook shows, and its limits
- **Shows:** how Haiku, Sonnet and Jev do on the same 160 chats, inputs and answers as the tuned model, how much they vary from run to run, what each costs per chat, and how fast each answers.
- **Limits:**
  - The team tuned the router prompt while looking at the 30 validation chats, which are part of these 160, so Haiku's and Sonnet's pool scores are slightly optimistic. The 40 test chats, scored once in the final run, are the clean comparison.
  - The tuned model shown is notebook 02's model (original recipe). Notebook 03 may change which tuned model goes forward; the final run compares the chosen one.
  - Jev's questions were written once from the team's prompt wording and not tuned (one attempt); the prompt for Haiku and Sonnet went through three team versions.
  - Speed here includes the internet round trip from Kaggle's servers; production speed depends on where the system runs.
  - Jev is in early access and its address has "alpha" in it, so its behaviour or address may change.
  - Still one generator behind all the chats.